In [9]:
import os
import numpy as np
import matplotlib.pyplot as plt
from minisom import MiniSom
from sklearn import datasets
from sklearn.preprocessing import StandardScaler

# Create output directory for snapshots if it does not exist
output_dir = "som_snapshots"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# -----------------------------
# 1. Data Preparation: Load and standardize the Iris dataset
# -----------------------------
iris = datasets.load_iris()
data = iris.data              # 4-dimensional features
target = iris.target          # Classes: 0, 1, 2
target_names = iris.target_names  # e.g., 'setosa', 'versicolor', 'virginica'

scaler = StandardScaler()
data_std = scaler.fit_transform(data)

# -----------------------------
# 2. Initialize SOM Parameters and the SOM
# -----------------------------
som_rows, som_cols = 20, 20         # Define the SOM grid dimensions
input_len = data_std.shape[1]       # Number of features (4 for Iris)
initial_sigma = 1.0                 # Initial neighborhood radius
initial_learning_rate = 0.5         # Initial learning rate
T = 1000                            # Total number of iterations
snapshot_interval = 100             # Save a snapshot every 100 iterations

# Initialize the SOM
som = MiniSom(som_rows, som_cols, input_len,
              sigma=initial_sigma, learning_rate=initial_learning_rate,
              neighborhood_function='gaussian', random_seed=42)
# Initialize weights using PCA for faster convergence
som.pca_weights_init(data_std)

# -----------------------------
# 3. Manual Training Loop with Process Visualization
# -----------------------------
for t in range(T + 1):
    # Select a random sample from the training set
    x = data_std[np.random.randint(0, len(data_std))]
    
    # Find the Best Matching Unit (BMU) for the sample x
    winner = som.winner(x)
    
    # Decay the learning rate and sigma over time
    lr = initial_learning_rate * np.exp(-t / T)
    sig = initial_sigma * np.exp(-t / T)
    
    # Update the weights of each neuron in the SOM grid
    for i in range(som_rows):
        for j in range(som_cols):
            neuron_location = np.array([i, j])
            # Compute squared Euclidean distance between the neuron and the BMU (in grid coordinates)
            distance_sq = np.sum((neuron_location - np.array(winner)) ** 2)
            # Gaussian neighborhood function:
            h = np.exp(-distance_sq / (2 * (sig ** 2)))
            # Update the neuron's weight vector:
            som._weights[i, j, :] += lr * h * (x - som._weights[i, j, :])
    
    # Every 'snapshot_interval' iterations, visualize and save the current state of the SOM
    if t % snapshot_interval == 0:
        # Compute the U-Matrix from the current SOM weights.
        # The U-Matrix reflects the average distance between a neuron and its neighbors.
        u_matrix = som.distance_map()
        
        plt.figure(figsize=(12, 10))
        # Plot the U-Matrix heatmap with a transparent background
        plt.pcolor(u_matrix.T, cmap='coolwarm', alpha=0.9)
        plt.colorbar(label='Average Distance')
        plt.title(f'SOM Training Process at Iteration {t}', fontsize=16)
        
        # Define different markers and colors for the Iris classes
        markers = ['o', 's', 'D']
        colors = ['r', 'g', 'b']
        
        # Plot the BMU for each training sample on the SOM grid
        for idx, sample in enumerate(data_std):
            bmu = som.winner(sample)
            # Offset by 0.5 to center the marker in the cell
            plt.plot(bmu[0] + 0.5, bmu[1] + 0.5, markers[target[idx]],
                     markerfacecolor='None', markeredgecolor=colors[target[idx]],
                     markersize=12, markeredgewidth=2)
        
        # Add legend entries for each Iris class
        for class_idx, name in enumerate(target_names):
            plt.scatter([], [], marker=markers[class_idx],
                        edgecolors=colors[class_idx], facecolors='None',
                        s=100, label=name)
        #plt.legend(loc='upper right')
        
        #plt.tight_layout()
        # Save the snapshot with a transparent background
        filename = os.path.join(output_dir, f'som_snapshot_{t:04d}.png')
        plt.savefig(filename, transparent=True)
        plt.close()


/Users/ada/Desktop/SGX/.conda/lib/python3.11/site-packages/minisom.py:447: ComplexWarning: Casting complex values to real discards the imaginary part
  self._weights[i, j] = c1*pc[pc_order[0]] + \
